# Phase 2 - Part 2: Wind farm simulator & optimisation (intro)

Phase 2 fixes **55 × IEA 22 MW** (1210 MW). Bastankhah wake, 15 × 15 km box (|x|,|y| ≤ 7500 m), ≥ 5D spacing, fixed-bottom depth ≤ 50 m. Ranked on **capacity factor**. The notebook walks the simulator + each optimisation axis; in practice you only tune **siting + layout**.

## 1. Setup

In [ ]:
import os, sys, warnings
from pathlib import Path
try:
    _here = Path(__vsc_ipynb_file__).resolve().parent   # VS Code sets this
except NameError:
    _here = Path.cwd().resolve()
_root = next(d for d in [_here, *_here.parents]
             if (d / 'part0_dataset_setup' / 'target_loader.py').exists())
os.chdir(_root)
sys.path[:0] = ['.', 'part0_dataset_setup', 'part1_forecast',
                'part2_siting', 'part3_economics']
warnings.filterwarnings('ignore')
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import synthetic_generator as sg
import synth_wind

from turbines_catalog import (
    CATALOG, get_spec, load_turbine, summary_table,
)
from wind_farm_simulator import (
    FarmLayout, WindSeries, derive_ti_per_sector,
    grid_layout, validate_layout,
    simulate_day, simulate_year,
)
from optimization import (
    WindAtPointCache, FarmConfig, evaluate_config,
    optimize_placement, optimize_layout, optimize_joint,
)

plt.rcParams.update({'figure.dpi': 100, 'figure.figsize': (10, 5),
                     'font.size': 10, 'axes.grid': True, 'axes.spines.top': False, 'axes.spines.right': False})
print('Phase 2 simulator kit loaded.')

## 2. Turbine

Phase 2 uses a single turbine - the **IEA 22 MW** (284 m rotor, 170 m hub). The other reference turbines are greyed for context only.

In [ ]:
df_cat = summary_table()
df_cat

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ws = np.linspace(0, 30, 200)
for spec in CATALOG.values():
    wt = load_turbine(spec.key)
    used = spec.key == 'IEA_22MW'
    ax.plot(ws, wt.power(ws) / 1e6,
            lw=3 if used else 1.2, color='tab:red' if used else 'lightgrey',
            zorder=3 if used else 1,
            label=f'{spec.name} ({spec.rated_power_mw:.0f} MW, D={spec.diameter_m:.0f} m)' + (' - Phase 2' if used else ''))
ax.set(xlabel='Wind speed at hub (m/s)', ylabel='Single-turbine power (MW)',
       title='Power curves - Phase 2 uses the IEA 22 MW (others for context)', xlim=(0, 27))
ax.legend(loc='upper left', fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 3. Wind Data - Synthetic North Sea Year

Hub height 170 m (IEA 22 MW), Dogger Bank-like centre (54.5N, 2E).

In [ ]:
hist = sg.load_coarse_history()
synth = sg.bootstrap_year(hist, block_days=14, seed=0)
print(f'Synthetic year: {synth.times.size} steps over {synth.lat.size} sea-grid cells')

wind_cache = synth_wind.SynthWindCache(synth, hub_height_m=170.0)

DOGGER_LAT, DOGGER_LON = 54.5, 2.0
ws = wind_cache.get(DOGGER_LAT, DOGGER_LON)
print(f'\nDogger Bank-like centre (snapped grid): {ws.n_steps} steps over {ws.duration_hours/24:.0f} days')
print(f'  mean ws @ 170 m: {ws.df["ws"].mean():.2f} m/s   std: {ws.df["ws"].std():.2f}')

In [ ]:
fig = plt.figure(figsize=(13, 5))
ax1 = fig.add_subplot(1, 2, 1, projection='polar')
n_sectors = 12
sw = 360 / n_sectors
centres = np.arange(n_sectors) * sw
freqs = np.zeros(n_sectors)
mean_ws = np.zeros(n_sectors)
for i, c in enumerate(centres):
    lo = (c - sw/2) % 360; hi = (c + sw/2) % 360
    if lo < hi: m = (ws.df['wd'] >= lo) & (ws.df['wd'] < hi)
    else:       m = (ws.df['wd'] >= lo) | (ws.df['wd'] < hi)
    freqs[i] = m.mean(); mean_ws[i] = float(ws.df.loc[m, 'ws'].mean()) if m.sum() > 0 else 0
theta = np.deg2rad(centres)
ax1.bar(theta, freqs * 100, width=np.deg2rad(sw)*0.9, color=plt.cm.viridis(mean_ws / mean_ws.max()), edgecolor='white')
ax1.set_theta_zero_location('N'); ax1.set_theta_direction(-1)
ax1.set_title('Wind rose - frequency (%) & mean speed (colour)\nsynthetic North Sea year', va='bottom', y=1.08)
ax1.grid(alpha=0.3)

ax2 = fig.add_subplot(1, 2, 2)
ax2.plot(ws.df['time'], ws.df['ws'], lw=0.5, color='steelblue')
ax2.set(xlabel='Date', ylabel='Wind speed @ 170m (m/s)', title='Time series - synthetic year')
ax2.axhline(ws.df['ws'].mean(), ls='--', color='black', alpha=0.6, label=f'mean = {ws.df["ws"].mean():.2f} m/s')
ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 4. Turbulence Intensity per Direction Sector

In [ ]:
centres, ti = derive_ti_per_sector(ws.df['ws'].values, ws.df['wd'].values)
fig = plt.figure(figsize=(7, 5))
ax = fig.add_subplot(111, projection='polar')
theta = np.deg2rad(centres)
ax.bar(theta, ti * 100, width=np.deg2rad(30)*0.9, color=plt.cm.coolwarm(ti / ti.max()), edgecolor='white')
ax.set_theta_zero_location('N'); ax.set_theta_direction(-1)
ax.set_title('TI per 30° sector (%)', va='bottom', y=1.08)
for c, t in zip(centres, ti):
    ax.text(np.deg2rad(c), t*100 + 0.4, f'{t*100:.1f}%', ha='center', fontsize=8)
plt.tight_layout(); plt.show()
print(f'TI range (offshore): {ti.min()*100:.1f}% - {ti.max()*100:.1f}%')

## 5. Baseline Farm - Layout & Validation

In [ ]:
BOX_M = 15_000
MAX_TURBINES = 55
MIN_SPACING_D = 5

spec = get_spec('IEA_22MW')
x, y = grid_layout(n_turbines=55, spacing_d=7, diameter_m=spec.diameter_m, rotation_deg=0)
ok, errs = validate_layout(x, y, box_size_m=BOX_M, max_turbines=MAX_TURBINES,
                            min_spacing_d=MIN_SPACING_D, diameter_m=spec.diameter_m)
print(f'Layout valid: {ok}  errors: {errs}')

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(x/1000, y/1000, s=120, marker='2', c='black', label=f'55 × IEA 22 MW')
from matplotlib.patches import Rectangle
ax.add_patch(Rectangle((-BOX_M/2/1000, -BOX_M/2/1000), BOX_M/1000, BOX_M/1000,
                        fill=False, edgecolor='red', ls='--', lw=2, label=f'{BOX_M/1000:g}×{BOX_M/1000:g} km box'))
ax.set(xlabel='x (km)', ylabel='y (km)', title='Baseline layout - 5×5 grid, 7D spacing')
ax.set_aspect('equal'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 6. One-Day Simulation

In [ ]:
doy = 100
day_mask = ws.df['time'].dt.dayofyear == doy
day_wind = WindSeries(pd.DataFrame({
    'time': ws.df.loc[day_mask, 'time'],
    'ws':   ws.df.loc[day_mask, 'ws'],
    'wd':   ws.df.loc[day_mask, 'wd'],
}))
turbine = load_turbine('IEA_22MW')
layout = FarmLayout(x_m=x, y_m=y, turbine=turbine)
day_res = simulate_day(layout, day_wind)

fig, ax = plt.subplots(figsize=(11, 4.5))
for i in range(layout.n_turbines):
    ax.plot(day_res.times, day_res.per_turbine_power_mw[:, i], lw=0.5, alpha=0.4, color='steelblue')
ax.plot(day_res.times, day_res.farm_power_mw, lw=2.5, color='black', label=f'Farm total - {layout.n_turbines} turbines')
ax.set(xlabel='Time', ylabel='Power (MW)',
       title=f'1-day simulation - DOY {doy}\nMean farm power: {day_res.farm_power_mw.mean():.0f} MW, peak: {day_res.farm_power_mw.max():.0f} MW')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 7. One-Year Simulation - AEP, Capacity Factor, Wake Loss

In [ ]:
t0 = time.time()
year_res = simulate_year(layout, ws)
print(f'1-year simulation: {time.time()-t0:.1f}s, {ws.n_steps:,} timesteps')
print(f'  rated capacity: {year_res.rated_capacity_mw:.0f} MW')
print(f'  AEP:            {year_res.aep_gwh:.0f} GWh/year')
print(f'  capacity factor:{year_res.capacity_factor*100:.1f}%')
print(f'  wake loss:      {year_res.wake_loss_fraction*100:.2f}%')
print(f'  mean farm power:{year_res.farm_power_mw.mean():.1f} MW')

df_year = pd.DataFrame({'time': year_res.times, 'farm_mw': year_res.farm_power_mw})
df_year['time'] = pd.to_datetime(df_year['time'])
monthly_cf = df_year.set_index('time')['farm_mw'].resample('ME').mean() / year_res.rated_capacity_mw * 100
fig, ax = plt.subplots(figsize=(10, 3.5))
monthly_cf.plot(kind='bar', ax=ax, color=plt.cm.RdYlGn(monthly_cf.values / 100), edgecolor='white')
ax.axhline(year_res.capacity_factor * 100, ls='--', color='red', lw=1.5, label=f'Annual CF = {year_res.capacity_factor*100:.1f}%')
ax.set(xlabel='Month', ylabel='Capacity factor (%)', title=f'Monthly capacity factor - 55 × IEA 22 MW = {year_res.rated_capacity_mw:.0f} MW @ Dogger Bank')
ax.set_xticklabels([t.strftime('%b') for t in monthly_cf.index], rotation=0)
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

## 8. Optimisation 1 - Farm Placement (centre lat/lon)

In [ ]:
t0 = time.time()
res_place = optimize_placement(
    layout_x_m=x, layout_y_m=y, turbine_key='IEA_22MW', wind_cache=wind_cache,
    lat_bounds=(53.5, 55.5), lon_bounds=(0.5, 4.0),
    max_iter=15, seed=42,
)
print(f'Best centre: ({res_place.best_config.centre_lat:.3f}°N, {res_place.best_config.centre_lon:.3f}°E)')
print(f'  AEP: {res_place.best_aep_gwh:.0f} GWh   CF: {res_place.best_capacity_factor*100:.1f}%')
print(f'  vs baseline (Dogger Bank): {res_place.best_aep_gwh - year_res.aep_gwh:+.0f} GWh')

log_df = pd.DataFrame([e for e in res_place.log if 'aep_gwh' in e])
import cartopy.crs as ccrs, cartopy.feature as cfeature
fig, ax = plt.subplots(figsize=(8, 6), subplot_kw={'projection': ccrs.PlateCarree()})
ax.add_feature(cfeature.LAND, facecolor='#eae6da', zorder=0)
ax.add_feature(cfeature.COASTLINE, lw=0.8, zorder=2)
ax.set_extent([0.0, 4.5, 53.2, 55.8], crs=ccrs.PlateCarree())
sc = ax.scatter(log_df['lon'], log_df['lat'], c=log_df['aep_gwh'], cmap='YlOrRd', s=70,
                edgecolor='white', lw=0.4, transform=ccrs.PlateCarree(), zorder=3)
ax.scatter(res_place.best_config.centre_lon, res_place.best_config.centre_lat, marker='*', s=380,
           c='red', edgecolor='black', lw=1.5, transform=ccrs.PlateCarree(), zorder=4,
           label=f'best: AEP={res_place.best_aep_gwh:.0f} GWh')
ax.scatter(DOGGER_LON, DOGGER_LAT, marker='o', s=120, facecolor='none', edgecolor='black', lw=1.5,
           transform=ccrs.PlateCarree(), zorder=4, label='Dogger Bank baseline')
gl = ax.gridlines(draw_labels=True, alpha=0.3); gl.top_labels = False; gl.right_labels = False
plt.colorbar(sc, ax=ax, label='AEP (GWh/year)', shrink=0.85)
ax.set_title('Placement search - AEP across the zone')
ax.legend(loc='lower right'); plt.tight_layout(); plt.show()

## 9. Optimisation 2 - Layout (Spacing x Rotation)

In [ ]:
res_layout = optimize_layout(
    centre_lat=DOGGER_LAT, centre_lon=DOGGER_LON, turbine_key='IEA_22MW', wind_cache=wind_cache,
    n_turbines=55, box_size_m=BOX_M, max_turbines=MAX_TURBINES, min_spacing_d=MIN_SPACING_D,
    max_iter=15, seed=42,
)
print(f'Best layout: spacing={res_layout.log[-1].get("spacing_d",0):.2f}D, '
      f'rotation={res_layout.log[-1].get("rotation",0):.1f}°')
print(f'  AEP: {res_layout.best_aep_gwh:.0f} GWh   wake loss: {res_layout.best_wake_loss*100:.2f}%')
print(f'  vs 7D baseline:  {res_layout.best_aep_gwh - year_res.aep_gwh:+.0f} GWh')

log_df = pd.DataFrame([e for e in res_layout.log if 'aep_gwh' in e])
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].scatter(log_df['spacing_d'], log_df['wake_loss']*100, c=log_df['aep_gwh'], cmap='YlOrRd', s=40, edgecolor='white')
axes[0].set(xlabel='Spacing (rotor diameters)', ylabel='Wake loss (%)',
            title='Spacing vs wake loss')
axes[1].scatter(log_df['rotation'], log_df['aep_gwh'], c=log_df['wake_loss']*100, cmap='RdYlGn_r', s=40, edgecolor='white')
axes[1].set(xlabel='Rotation (°)', ylabel='AEP (GWh/year)', title='Rotation vs AEP (colour = wake loss %)')
for a in axes: a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 10. Joint optimisation - placement × layout (turbine fixed to IEA 22 MW)

In [ ]:
t0 = time.time()
res_joint = optimize_joint(
    initial_centre_lat=DOGGER_LAT, initial_centre_lon=DOGGER_LON,
    initial_turbine_key='IEA_22MW', n_turbines=55, wind_cache=wind_cache,
    fix_turbine=True,
    lat_bounds=(53.5, 55.5), lon_bounds=(0.5, 4.0),
    box_size_m=BOX_M, max_turbines=MAX_TURBINES, min_spacing_d=MIN_SPACING_D,
    n_outer_iters=2, max_iter_per_axis=10, seed=42,
)
print(f'JOINT result ({time.time()-t0:.1f}s):')
print(f'  centre:  ({res_joint.best_config.centre_lat:.3f}°N, {res_joint.best_config.centre_lon:.3f}°E)')
print(f'  turbine: {res_joint.best_config.turbine_key}')
print(f'  AEP:     {res_joint.best_aep_gwh:.0f} GWh   CF: {res_joint.best_capacity_factor*100:.1f}%')
print(f'  vs baseline: {res_joint.best_aep_gwh - year_res.aep_gwh:+.0f} GWh '
      f'({100*(res_joint.best_aep_gwh - year_res.aep_gwh)/year_res.aep_gwh:+.0f}%)')

labels = ['Baseline', 'Placement', 'Layout', 'Joint']
aeps = [year_res.aep_gwh, res_place.best_aep_gwh, res_layout.best_aep_gwh, res_joint.best_aep_gwh]
fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#888', '#4477AA', '#66CC66', '#CC4444']
bars = ax.bar(labels, aeps, color=colors, edgecolor='white')
for bar, v in zip(bars, aeps):
    ax.annotate(f'{v:.0f}', (bar.get_x()+bar.get_width()/2, bar.get_height()), ha='center', va='bottom', fontsize=10)
ax.set(ylabel='AEP (GWh/year)', title='AEP gain from each optimisation axis')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

## 11. Submission Format

In [ ]:
import json
submission = {
    'team': 'demo_team',
    'farm_centre_lat': float(res_joint.best_config.centre_lat),
    'farm_centre_lon': float(res_joint.best_config.centre_lon),
    'turbine_key': res_joint.best_config.turbine_key,
    'layout_x_m': res_joint.best_config.layout_x_m.round(2).tolist(),
    'layout_y_m': res_joint.best_config.layout_y_m.round(2).tolist(),
    'reported': {
        'aep_gwh':            round(float(res_joint.best_aep_gwh), 2),
        'capacity_factor':    round(float(res_joint.best_capacity_factor), 4),
        'wake_loss_fraction': round(float(res_joint.best_wake_loss), 4),
    },
}
print(json.dumps(submission, indent=2)[:1500])